# problem : 

A,B,C,D = ?

(1) A + B = 9

(2) C - D = 1

(3) A + C = 8

(4) B - D = 2

# answer : 
A = 5

B = 4

C = 3

D = 2


## Using PYTORCH

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import math
import torch

## first approach

In this approach, the loss function is constructed directly from the given equations, without enforcing the integer constraint on the variables.

```python

loss = (A+B - 9)**2 + (C-D - 1)**2 + (A+C - 8)**2 + (B-D - 2)**2

```

Here, the loss is defined as the **sum of squared errors (SSE)** of the four equations. The optimizer tries to minimize this loss so that the equations are satisfied, but the variables are not restricted to integer values, so the solution may be fractional.

In [60]:

A = torch.tensor(1.0, requires_grad=True)
B = torch.tensor(1.0, requires_grad=True)
C = torch.tensor(1.0, requires_grad=True)
D = torch.tensor(1.0, requires_grad=True)

optimizer = torch.optim.Adam([A, B, C, D], lr=0.1)

StepCount = 600
for step in range(StepCount):
    optimizer.zero_grad()

    loss = (A+B - 9)**2 + (C-D - 1)**2 + (A+C - 8)**2 + (B-D - 2)**2

    loss.backward()
    optimizer.step()

# print("Estimated A = ", A.item())
# print("Estimated B = ", B.item())
# print("Estimated C = ", C.item())
# print("Estimated D = ", D.item())
# print("============================================")
print("Estimated A = ", round( A.item() ,3) )
print("Estimated B = ", round( B.item() ,3) )
print("Estimated C = ", round( C.item() ,3) )
print("Estimated D = ", round( D.item() ,3) )
print("============================================")
print("A+B = ", round( A.item() + B.item() ) )
print("C-D = ", round( C.item() - D.item() ) )
print("A+C = ", round( A.item() + C.item() ) )
print("B-D = ", round( B.item() - D.item() ) )

Estimated A =  4.579
Estimated B =  4.421
Estimated C =  3.421
Estimated D =  2.421
A+B =  9
C-D =  1
A+C =  8
B-D =  2


## second approach

In this approach, I enforce the integer constraint alongside the Sum of Squared Errors (SSE) loss for the equations. This uses a regularization term to push the continuous variables towards the nearest integers.

#### SSE Loss for the Equation Constraints ($L_{eq}$)
```python

eq_loss = (A+B-9)**2 + (C-D-1)**2 +  (A+C-8)**2 +  (B-D-2)**2

```

#### Integer Constraint Loss ($L_{int}$)
```python

int_loss = torch.sin(math.pi*A)**2 +  torch.sin(math.pi*B)**2 + torch.sin(math.pi*C)**2 +  torch.sin(math.pi*D)**2

```

**Note** : The term $\sin^2(\pi x)$ is used as a smooth penalty. It equals zero if and only if $x$ is an integer, making it an effective differentiable surrogate for an integer constraint.


#### Combined Total Loss

The final loss is a weighted sum of the constraint satisfaction and the integer regularization.

```python

    loss = eq_loss + lambda_ * int_loss

```

In [61]:

A = torch.randn(1, requires_grad=True)
B = torch.randn(1, requires_grad=True)
C = torch.randn(1, requires_grad=True)
D = torch.randn(1, requires_grad=True)

optimizer = torch.optim.Adam([A,B,C,D], lr=0.1)

lambda_ = 0.5

for i in range(200):
    optimizer.zero_grad()

    eq_loss = (A+B-9)**2 + (C-D-1)**2 +  (A+C-8)**2 +  (B-D-2)**2

    int_loss = torch.sin(math.pi*A)**2 +  torch.sin(math.pi*B)**2 + torch.sin(math.pi*C)**2 +  torch.sin(math.pi*D)**2

    loss = eq_loss + lambda_ * int_loss
    loss.backward()
    optimizer.step()

print("Estimated A = ", round( A.item() ,2) )
print("Estimated B = ", round( B.item() ,2) )
print("Estimated C = ", round( C.item() ,2) )
print("Estimated D = ", round( D.item() ,2) )
print("============================================")
print("A+B = ", round( A.item() + B.item() ) )
print("C-D = ", round( C.item() - D.item() ) )
print("A+C = ", round( A.item() + C.item() ) )
print("B-D = ", round( B.item() - D.item() ) )

Estimated A =  5.0
Estimated B =  4.0
Estimated C =  3.0
Estimated D =  2.0
A+B =  9
C-D =  1
A+C =  8
B-D =  2
